![title](imagens/projeto18.jpg)

# Projeto 18 - Previsão do Tempo de Entrega de Alimentos
-------

Prever o tempo de entrega do seu pedido é uma tarefa desafiadora para todo serviço de entrega de alimentos, como iFood, Zomato e Swiggy.

Uma das melhores estratégias para prever o tempo de entrega é calculando a distância entre o ponto de retirada do pedido e o ponto de entrega do pedido. E então prever o tempo de entrega com base em quanto tempo seus parceiros de entrega levaram para entregar pedidos no passado para a mesma distância.

O conjunto de dados que você tem aqui é uma versão limpa do conjunto de dados original enviado por Gaurav Malik no Kaggle (Você pode acessar todas as informações no seguinte link:https://www.kaggle.com/datasets/gauravmalik26/food-delivery-dataset). Abaixo estão todas as características no conjunto de dados:

- ID: número do pedido
- Delivery_person_ID: número de identificação do parceiro de entrega
- Delivery_person_Age: idade do parceiro de entrega
- Delivery_person_Ratings: avaliações do parceiro de entrega com base em entregas anteriores
- Restaurant_latitude: A latitude do restaurante
- Restaurant_longitude: A longitude do restaurante
- Delivery_location_latitude: A latitude do local de entrega
- Delivery_location_longitude: A longitude do local de entrega
- Type_of_order: O tipo de refeição encomendada pelo cliente
- Type_of_vehicle: O tipo de veículo que o parceiro de entrega utiliza
- Time_taken(min): O tempo gasto pelo parceiro de entrega para completar o pedido

Você precisa prever o tempo de entrega com base na distância percorrida pelo parceiro de entrega para entregar o pedido.

Então, para esta tarefa, precisamos de um conjunto de dados contendo informações sobre o tempo que os parceiros de entrega levam para entregar comida do restaurante para o local de entrega. 
Nos recursos dessa aula você encontrará o conjunto de dados no arquivo: **deliverytime.txt**, dentro do diretório ``dados``.

In [ ]:
# Versão da Linguagem Python
from platform import python_version
print('Versão da Linguagem Python usada neste Projeto no Jupyter Notebook:', python_version())

In [ ]:
# Para atualizar um pacote, execute o comando abaixo no terminal ou prompt de comando:
# pip install -U nome_pacote

# Para instalar a versão exata de um pacote, execute o comando abaixo no terminal ou prompt de comando:
# !pip install nome_pacote==versão_desejada

# Depois de instalar ou atualizar o pacote, reinicie o jupyter notebook.

# Instala o pacote watermark. 
# Esse pacote é usado para gravar as versões de outros pacotes usados neste jupyter notebook.
# !pip install -q -U watermark

# Neste projeto vamos utilizar redes neurais... para tanto, vamos precisar do pacote Keras.
# Se não tiver o Keras e o Tensorflow não estiverem instalados... tem que ser instalados...
# !pip install tensorflow
# !pip install keras

In [ ]:
# Vamos importar as biblitecas que vamos precisar para este projeto
import pandas as pd
import numpy as np
import plotly.express as px

# importação dos módulos de redes neurais do Keras
import keras
from keras.models import Sequential
from keras.layers import Dense, LSTM

import tensorflow

# importando funções para preparação dos dados para o modelo e verificação das métricas
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score, mean_absolute_percentage_error

# para evitar mensagens de alerta/warnings.
import warnings
warnings.filterwarnings("ignore")

# Carregar o módulo de funções para limpeza de dados
from limpeza_dados import *

In [ ]:
# Versões dos pacotes usados neste jupyter notebook
%reload_ext watermark
%watermark -a "pyPRO - Seja um Profissional Python!" --iversions

### Carregando o Dataset

In [ ]:
# Vamos carregar o dataset
df = pd.read_csv("dados/deliverytime.txt")
print(df.head())

In [ ]:
# Vamos ver as informações dos campos...
df.info()

In [ ]:
# verificando se existe algum valor nulo
calcular_porcentagem_valores_ausentes(df)

Não existem valores em branco ou nulos.  Vamos a diante.


## Calculando a Distância Entre Duas Latitudes e Longitudes

O conjunto de dados não possui nenhuma característica que mostre a diferença entre o restaurante e o local de entrega. Tudo o que temos são os pontos de latitude e longitude do restaurante e do local de entrega. Podemos usar a fórmula de Haversine para calcular a distância entre dois locais com base em suas latitudes e longitudes.

Abaixo está como podemos encontrar a distância entre o restaurante e o local de entrega com base em suas latitudes e longitudes usando a fórmula de Haversine (você pode ver mais detalhes em: https://en.wikipedia.org/wiki/Haversine_formula)


distancia = 2 * raio_da_terra * arcsin(sqrt(sin²((latitude_entrega - latitude_restaurante) / 2) + cos(latitude_restaurante) * cos(latitude_entrega) * sin²((longitude_entrega - longitude_restaurante) / 2)))


In [ ]:
# Set the earth's radius (in kilometers)
R = 6371

# Convert degrees to radians
def deg_to_rad(degrees):
    return degrees * (np.pi/180)

# Function to calculate the distance between two points using the haversine formula
def distcalculate(lat1, lon1, lat2, lon2):
    d_lat = deg_to_rad(lat2-lat1)
    d_lon = deg_to_rad(lon2-lon1)
    a = np.sin(d_lat/2)**2 + np.cos(deg_to_rad(lat1)) * np.cos(deg_to_rad(lat2)) * np.sin(d_lon/2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))
    return R * c
  
# Calculate the distance between each pair of points
df['distance'] = np.nan

for i in range(len(df)):
    df.loc[i, 'distance'] = distcalculate(df.loc[i, 'Restaurant_latitude'], 
                                        df.loc[i, 'Restaurant_longitude'], 
                                        df.loc[i, 'Delivery_location_latitude'], 
                                        df.loc[i, 'Delivery_location_longitude'])

Agora calculamos a distância entre o restaurante e o local de entrega. Também adicionamos uma nova característica no conjunto de dados como distância. Vamos dar uma olhada no conjunto de dados novamente:

In [ ]:
print(df.head())

## Exploração de Dados
Agora vamos explorar os dados para encontrar relacionamentos entre as características. Vou começar olhando para o relacionamento entre a distância e o tempo levado para entregar a comida:


In [ ]:
figure = px.scatter(data_frame = df, 
                    x="distance",
                    y="Time_taken(min)", 
                    size="Time_taken(min)", 
                    trendline="ols", 
                    title = "Relationship Between Distance and Time Taken")
figure.show()

Existe uma relação consistente entre o tempo levado e a distância percorrida para entregar a comida. Isso significa que a maioria dos parceiros de entrega entrega comida dentro de 25-30 minutos, independentemente da distância.

Agora vamos dar uma olhada na relação entre o tempo levado para entregar a comida e a idade do parceiro de entrega:


In [ ]:
figure = px.scatter(data_frame = df, 
                    x="Delivery_person_Age",
                    y="Time_taken(min)", 
                    size="Time_taken(min)", 
                    color = "distance",
                    trendline="ols", 
                    title = "Relationship Between Time Taken and Age")
figure.show()

Existe uma relação linear entre o tempo levado para entregar a comida e a idade do parceiro de entrega. Isso significa que parceiros de entrega mais jovens levam menos tempo para entregar a comida em comparação com os parceiros mais velhos.

Agora vamos dar uma olhada na relação entre o tempo levado para entregar a comida e as avaliações do parceiro de entrega:


In [ ]:
figure = px.scatter(data_frame = df, 
                    x="Delivery_person_Ratings",
                    y="Time_taken(min)", 
                    size="Time_taken(min)", 
                    color = "distance",
                    trendline="ols", 
                    title = "Relationship Between Time Taken and Ratings")
figure.show()

Existe uma relação linear inversa entre o tempo levado para entregar a comida e as avaliações do parceiro de entrega. Isso significa que parceiros de entrega com avaliações mais altas levam menos tempo para entregar a comida em comparação com parceiros com avaliações mais baixas.

Agora vamos dar uma olhada se o tipo de comida pedida pelo cliente e o tipo de veículo usado pelo parceiro de entrega afetam o tempo de entrega ou não:


In [ ]:
fig = px.box(df, 
             x="Type_of_vehicle",
             y="Time_taken(min)", 
             color="Type_of_order")
fig.show()

Portanto, não há muita diferença entre o tempo levado pelos parceiros de entrega dependendo do veículo que estão dirigindo e do tipo de comida que estão entregando.

Então, as características que mais contribuem para o tempo de entrega de alimentos com base em nossa análise são:

- idade do parceiro de entrega
- avaliações do parceiro de entrega
- distância entre o restaurante e o local de entrega

A seguir, vamos treinar um modelo de Machine Learning para previsão do tempo de entrega de alimentos.

## Modelo de Previsão do Tempo de Entrega de Alimentos

Agora vamos treinar um modelo de Machine Learning usando uma rede neural LSTM para a tarefa de previsão do tempo de entrega de alimentos:


In [ ]:
#Fazendo a separação dos dados de treino e teste
x = np.array(df[["Delivery_person_Age", 
                   "Delivery_person_Ratings", 
                   "distance"]])
y = np.array(df[["Time_taken(min)"]])
xtrain, xtest, ytrain, ytest = train_test_split(x, y, 
                                                test_size=0.10, 
                                                random_state=42)

In [ ]:
# Criando o modelo de rede neural LSTM 
# Inicializa uma rede neural sequencial (uma camada após a outra)
model = Sequential()
# A primeira camada LSTM terá 128 neurônios // input_shape define a forma de entrada de dados. 
# Xtrain.shape representa o número de passos de tempo e o valor 1, representa o número de features
model.add(LSTM(128, return_sequences=True, input_shape= (xtrain.shape[1], 1)))
# Adiciona uma segunda camada com 64 neurônios. A opção False, faz com que essa camada adicione apenas uma única saída com um vetor de 64 dimensões.
model.add(LSTM(64, return_sequences=False))
# Adiciona uma primeira camada densa (totalmente conectada) com 25 unidades.
model.add(Dense(25))
# adiciona uma segunda camada densa com apenas uma unidade (saída)
model.add(Dense(1))
# a linha abaixo imprime um resumo do modelo.
model.summary()

In [ ]:
# Fazendo o treinamento do Modelo
# o primeiro parâmetro indica a função de ativação escolhida: Adam (Adaptive Moment Estimation). É eficiente em termos de memória e não precisa
# de muitos ajustes.  //  função de perda (loss) - Erro Médio Quadrático, é uma medida comum para problemas de regressão.
model.compile(optimizer='adam', loss='mean_squared_error')
# batch_size indica a quantidade de amostras de treinamento o modelo processa antes de atualizar os parâmetros.
# epochs indica o número de épocas ou seja a qtd de vezes que o modelo verá todo o conjunto de dados de treinamento.
model.fit(xtrain, ytrain, batch_size=1, epochs=9)

### Acurácia do Modelo

In [ ]:
# Avaliar o modelo com os dados de teste
loss = model.evaluate(xtest, ytest)
print(f'Loss (Erro Quadrático Médio): {loss}')

# Fazer previsões com os dados de teste
predictions = model.predict(xtest)

# Calcular métricas adicionais
mae = mean_absolute_error(ytest, predictions)
r2 = r2_score(ytest, predictions)
print(f'Erro Médio Absoluto (MAE): {mae}')
print(f'R² Score: {r2}')

# Calcular a acurácia do modelo
# Calcular MAPE
mape = mean_absolute_percentage_error(ytest, predictions)
# Calcular acurácia percentual
accuracy = 100 - mape * 100
print(f'Acurácia: {accuracy:.2f}%')

### Testando o Modelo

In [ ]:
print("Predição do Tempo de Entrega de Alimentos")
a = int(input("Idade do Entregador: "))
b = float(input("Avaliação das Entregas Anteriores (1-5): "))
c = int(input("Distância Total: "))

features = np.array([[a, b, c]])
tempo = model.predict(features)[0][0]
print(f"Previsão do Tempo de Entrega: {tempo:.2f} minutos")

Então, assim é como você pode usar Machine Learning para a tarefa de previsão do tempo de entrega de alimentos.

**Resumo**
Para prever o tempo de entrega de alimentos em tempo real, é necessário calcular a distância entre o ponto de preparo dos alimentos e o ponto de consumo dos alimentos. Após encontrar a distância entre o restaurante e os locais de entrega, é necessário encontrar relações entre o tempo levado pelos parceiros de entrega para entregar a comida no passado para a mesma distância.


## Fim do Projeto 18